# Agent Inheritance, Traits & Polymorphism

**Multigen SDK — Notebook 27**

---

## What this notebook covers

Python's object model gives Multigen agents a rich toolkit for code reuse and
extensibility. This notebook walks through every mechanism the SDK exposes:

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | InheritableAgent — subclassing and lifecycle hooks |
| 3 | Cooperative inheritance with super() chains |
| 4 | LoggingTrait — automatic DEBUG logging |
| 5 | RetryTrait — automatic retry on failure |
| 6 | CachingTrait — result memoisation |
| 7 | TimingTrait — _duration_ms injection |
| 8 | ValidatingTrait — required key enforcement |
| 9 | build_agent() — combine multiple traits with MRO control |
| 10 | @mixin decorator — apply traits without subclassing |
| 11 | @overload — single-dispatch by argument type |
| 12 | MultiMethod — n-ary dispatch with fallback |
| 13 | ShapeRegistry + PolymorphicAgent — runtime shape morphing |
| 14 | TypeAdapter — bidirectional type conversion registry |
| 15 | DynamicAgent.build() — runtime class construction |

All agents are pure in-memory. No external APIs required.

## Section 1 — Setup and Imports

The Multigen SDK places all inheritance and polymorphism utilities under a single
import path. We bring in:

- **`InheritableAgent`** — base class with lifecycle hooks (`pre_run`, `run`,
  `post_run`, `on_error`).
- **`Trait`** — base mixin class; all trait classes extend it.
- **`LoggingTrait`, `RetryTrait`, `CachingTrait`, `TimingTrait`,
  `ValidatingTrait`** — five ready-made traits.
- **`build_agent`** — factory that combines a base class with any number of traits
  via `type()`, respecting MRO.
- **`mixin`** — class decorator that splices traits into an existing class.
- **`overload`** — single-dispatch decorator (type → implementation).
- **`MultiMethod`** — n-ary dispatch table.
- **`implements`** — protocol-style interface checker.
- **`PolymorphicAgent`, `ShapeRegistry`** — runtime shape morphing.
- **`TypeAdapter`** — bidirectional type conversion registry.
- **`DynamicAgent`** — runtime class builder.

In [ ]:
import sys
import logging

sys.path.insert(0, '../sdk')

from multigen.inheritance import (
    InheritableAgent,
    Trait,
    LoggingTrait,
    RetryTrait,
    CachingTrait,
    TimingTrait,
    ValidatingTrait,
    build_agent,
    mixin,
    overload,
    MultiMethod,
    implements,
    PolymorphicAgent,
    ShapeRegistry,
    TypeAdapter,
    DynamicAgent,
)

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s %(name)s — %(message)s')

print('InheritableAgent :', InheritableAgent)
print('Trait            :', Trait)
print('LoggingTrait     :', LoggingTrait)
print('RetryTrait       :', RetryTrait)
print('CachingTrait     :', CachingTrait)
print('TimingTrait      :', TimingTrait)
print('ValidatingTrait  :', ValidatingTrait)
print('build_agent      :', build_agent)
print('mixin            :', mixin)
print('overload         :', overload)
print('MultiMethod      :', MultiMethod)
print('implements       :', implements)
print('PolymorphicAgent :', PolymorphicAgent)
print('ShapeRegistry    :', ShapeRegistry)
print('TypeAdapter      :', TypeAdapter)
print('DynamicAgent     :', DynamicAgent)
print()
print('All imports successful.')

## Section 2 — InheritableAgent: Subclassing and Lifecycle Hooks

`InheritableAgent` defines four overridable hooks that fire in a guaranteed order:

```
pre_run(ctx)  →  run(ctx)  →  post_run(ctx, result)  →  (on_error(ctx, exc) if exception)
```

- **`pre_run(ctx)`** — called before `run`. Mutate `ctx` for preprocessing, logging,
  or validation.
- **`run(ctx)`** — the main computation. Must return a dict.
- **`post_run(ctx, result)`** — called after `run` with the result dict. Mutate
  `result` for postprocessing.
- **`on_error(ctx, exc)`** — called if `run` raises; return a fallback dict or
  re-raise.

We track the call order explicitly using a list to prove the guarantee holds.

In [ ]:
call_log = []

class GreetAgent(InheritableAgent):
    def pre_run(self, ctx):
        call_log.append('pre_run')
        ctx['greeted_by'] = 'GreetAgent'

    def run(self, ctx):
        call_log.append('run')
        return {'greeting': f"Hello, {ctx.get('name', 'world')}!",
                'greeted_by': ctx['greeted_by']}

    def post_run(self, ctx, result):
        call_log.append('post_run')
        result['processed'] = True

    def on_error(self, ctx, exc):
        call_log.append('on_error')
        return {'error': str(exc), 'recovered': True}


agent = GreetAgent()
result = agent.execute({'name': 'Multigen'})

print('Call order:', call_log)
print('Result    :', result)
print()
assert call_log == ['pre_run', 'run', 'post_run'], f'Unexpected call order: {call_log}'
assert result['greeting'] == 'Hello, Multigen!'
assert result['processed'] is True
print('Call order verified: pre_run → run → post_run')

In [ ]:
# Verify on_error fires when run() raises

error_log = []

class BoomAgent(InheritableAgent):
    def pre_run(self, ctx):
        error_log.append('pre_run')

    def run(self, ctx):
        error_log.append('run')
        raise ValueError('intentional test error')

    def post_run(self, ctx, result):
        error_log.append('post_run')   # should NOT be called

    def on_error(self, ctx, exc):
        error_log.append('on_error')
        return {'error': str(exc), 'recovered': True}


boom = BoomAgent()
err_result = boom.execute({})

print('Call order on error:', error_log)
print('Error result       :', err_result)
print()
assert error_log == ['pre_run', 'run', 'on_error'], f'Unexpected error call order: {error_log}'
assert 'post_run' not in error_log, 'post_run must not fire when run() raises'
assert err_result['recovered'] is True
print('Error hook order verified: pre_run → run → on_error (post_run skipped)')

## Section 3 — Cooperative Inheritance with super() Chains

When multiple `InheritableAgent` subclasses sit in a diamond or chain, each one
calling `super().<hook>()` ensures every hook fires.  Python's MRO resolves the
chain deterministically.

We build:

```
InheritableAgent
    └── LoggingBase   (adds log lines to context)
            └── AuditBase  (adds audit stamp)
                    └── BusinessAgent  (does real work)
```

Each `pre_run` calls `super().pre_run(ctx)` first, then adds its own data.  The
call order at `BusinessAgent` is therefore:
`LoggingBase.pre_run → AuditBase.pre_run → BusinessAgent.pre_run`.

In [ ]:
chain_log = []

class LoggingBase(InheritableAgent):
    def pre_run(self, ctx):
        super().pre_run(ctx)
        chain_log.append('LoggingBase.pre_run')
        ctx.setdefault('_log', []).append('LoggingBase entered')

    def post_run(self, ctx, result):
        super().post_run(ctx, result)
        chain_log.append('LoggingBase.post_run')
        result.setdefault('_log', []).append('LoggingBase exited')


class AuditBase(LoggingBase):
    def pre_run(self, ctx):
        super().pre_run(ctx)
        chain_log.append('AuditBase.pre_run')
        ctx['_audit'] = 'AuditBase stamped'

    def post_run(self, ctx, result):
        super().post_run(ctx, result)
        chain_log.append('AuditBase.post_run')
        result['_audit_complete'] = True


class BusinessAgent(AuditBase):
    def pre_run(self, ctx):
        super().pre_run(ctx)
        chain_log.append('BusinessAgent.pre_run')

    def run(self, ctx):
        chain_log.append('BusinessAgent.run')
        return {'result': 'business logic done', 'audit': ctx.get('_audit')}

    def post_run(self, ctx, result):
        super().post_run(ctx, result)
        chain_log.append('BusinessAgent.post_run')


biz = BusinessAgent()
biz_result = biz.execute({'request': 'do something'})

print('MRO:', [cls.__name__ for cls in BusinessAgent.__mro__])
print()
print('Call chain log:')
for entry in chain_log:
    print(f'  {entry}')
print()
print('Result:', biz_result)
print()
assert biz_result['audit'] == 'AuditBase stamped'
assert biz_result.get('_audit_complete') is True
print('Cooperative super() chain verified.')

## Section 4 — LoggingTrait

`LoggingTrait` wraps an agent's `run()` with `logging.debug()` calls before and
after execution. It logs:

- The agent class name and input context keys.
- The output keys after a successful run.

We use `build_agent(MyAgent, LoggingTrait)` to compose the trait into a new class
without modifying `MyAgent` directly. We capture the log output using
`logging.handlers.MemoryHandler` to assert the log lines in the notebook.

In [ ]:
import logging

class SumAgent(InheritableAgent):
    """Adds two numbers from context."""
    def run(self, ctx):
        a = ctx.get('a', 0)
        b = ctx.get('b', 0)
        return {'sum': a + b}


# Capture log records in memory
log_records = []

class ListHandler(logging.Handler):
    def emit(self, record):
        log_records.append(record)

handler = ListHandler(level=logging.DEBUG)
root_logger = logging.getLogger()
root_logger.addHandler(handler)

# Build the composed class
LoggingSumAgent = build_agent(SumAgent, LoggingTrait)

print(f'Composed class name : {LoggingSumAgent.__name__}')
print(f'MRO                 : {[c.__name__ for c in LoggingSumAgent.__mro__]}')
print()

agent = LoggingSumAgent()
result = agent.execute({'a': 7, 'b': 13})

print(f'Result: {result}')
print()

# Filter to DEBUG records from this agent
debug_msgs = [r.getMessage() for r in log_records if r.levelno == logging.DEBUG]
print('DEBUG log lines captured:')
for msg in debug_msgs:
    print(f'  {msg}')
print()
assert result['sum'] == 20
assert any('before' in m.lower() or 'entering' in m.lower() or 'sumag' in m.lower()
           for m in debug_msgs), 'Expected a pre-run DEBUG log line'
print('LoggingTrait verified: DEBUG lines emitted before and after run().')

root_logger.removeHandler(handler)

## Section 5 — RetryTrait

`RetryTrait(max_retries=N)` retries a failing `run()` up to N times before
re-raising the final exception.

We write an agent whose `run()` increments an attempt counter and raises on the
first two attempts, then succeeds on the third. With `max_retries=3`, the trait
should absorb both failures and return the successful result.

In [ ]:
attempt_counter = {'count': 0}

class FlakyAgent(InheritableAgent):
    """Fails the first two calls, succeeds on the third."""
    def run(self, ctx):
        attempt_counter['count'] += 1
        if attempt_counter['count'] < 3:
            raise RuntimeError(f'Simulated failure #{attempt_counter["count"]}')
        return {'status': 'ok', 'attempts': attempt_counter['count']}


RetryFlakyAgent = build_agent(FlakyAgent, RetryTrait, retry_kwargs={'max_retries': 3})

print(f'Composed class: {RetryFlakyAgent.__name__}')
print(f'MRO           : {[c.__name__ for c in RetryFlakyAgent.__mro__]}')
print()

retry_agent = RetryFlakyAgent()
retry_result = retry_agent.execute({})

print(f'Result    : {retry_result}')
print(f'Total attempts made: {attempt_counter["count"]}')
print()
assert retry_result['status'] == 'ok'
assert attempt_counter['count'] == 3, 'Expected exactly 3 attempts'
print('RetryTrait verified: eventual success after 2 failures with max_retries=3.')

In [ ]:
# Verify that retries exhausted raises the exception

always_fail_counter = {'count': 0}

class AlwaysFailAgent(InheritableAgent):
    def run(self, ctx):
        always_fail_counter['count'] += 1
        raise ValueError('always fails')


RetryAlwaysFailAgent = build_agent(AlwaysFailAgent, RetryTrait, retry_kwargs={'max_retries': 2})

try:
    RetryAlwaysFailAgent().execute({})
    print('ERROR: should have raised')
except (ValueError, RuntimeError) as exc:
    print(f'Exception raised after retries exhausted: {type(exc).__name__}: {exc}')
    print(f'Total attempts made: {always_fail_counter["count"]}')
    assert always_fail_counter['count'] == 3, (
        f'Expected 3 attempts (1 initial + 2 retries), got {always_fail_counter["count"]}'
    )
    print('RetryTrait exhaustion verified: raised after 1 initial + 2 retry attempts.')

## Section 6 — CachingTrait

`CachingTrait` memoises `run()` results keyed on a subset of context fields.
Specify which context keys form the cache key via `cache_keys=[...]`.

Behaviour:
- First call with a given key: executes `run()`, stores result.
- Subsequent calls with the **same** key: returns cached result without calling
  `run()` again.
- Call with a **different** key: executes `run()` fresh.

We use a call counter to prove that `run()` is not called on a cache hit.

In [ ]:
cache_run_count = {'count': 0}

class ExpensiveLookupAgent(InheritableAgent):
    """Simulates an expensive database lookup."""
    def run(self, ctx):
        cache_run_count['count'] += 1
        query = ctx.get('query', '')
        # Simulate expensive work
        return {'result': f'data for [{query}]', 'computed_at_call': cache_run_count['count']}


CachedLookupAgent = build_agent(
    ExpensiveLookupAgent,
    CachingTrait,
    cache_kwargs={'cache_keys': ['query']},
)

cached_agent = CachedLookupAgent()

# First call — cache miss
r1 = cached_agent.execute({'query': 'what is 2+2'})
print(f'Call 1 (cache miss) : {r1}  |  run() calls so far: {cache_run_count["count"]}')

# Second call with same query — cache hit
r2 = cached_agent.execute({'query': 'what is 2+2'})
print(f'Call 2 (cache hit)  : {r2}  |  run() calls so far: {cache_run_count["count"]}')

# Third call with different query — cache miss
r3 = cached_agent.execute({'query': 'capital of France'})
print(f'Call 3 (cache miss) : {r3}  |  run() calls so far: {cache_run_count["count"]}')

# Fourth call with first query again — cache hit
r4 = cached_agent.execute({'query': 'what is 2+2'})
print(f'Call 4 (cache hit)  : {r4}  |  run() calls so far: {cache_run_count["count"]}')

print()
assert cache_run_count['count'] == 2, f'Expected 2 real run() calls, got {cache_run_count["count"]}'
assert r1 == r2 == r4, 'Cache hits must return identical results'
assert r1['result'] != r3['result'], 'Different queries must produce different results'
print('CachingTrait verified: run() called only on cache misses.')

## Section 7 — TimingTrait

`TimingTrait` measures wall-clock time spent in `run()` and injects `_duration_ms`
into the result dict.  This lets downstream steps and observability tools inspect
per-agent latency without modifying agent code.

In [ ]:
import time

class SlowAgent(InheritableAgent):
    """Sleeps briefly to produce a measurable duration."""
    def run(self, ctx):
        time.sleep(0.05)   # 50 ms
        return {'answer': 42}


TimedSlowAgent = build_agent(SlowAgent, TimingTrait)

timed_agent = TimedSlowAgent()
timed_result = timed_agent.execute({})

print(f'Result: {timed_result}')
print()
assert '_duration_ms' in timed_result, '_duration_ms must be injected by TimingTrait'
duration = timed_result['_duration_ms']
print(f'_duration_ms = {duration:.2f} ms')
assert duration >= 40, f'Expected >= 40 ms (50 ms sleep), got {duration:.2f} ms'
assert timed_result['answer'] == 42
print('TimingTrait verified: _duration_ms injected and reflects actual sleep time.')

## Section 8 — ValidatingTrait

`ValidatingTrait` checks that all keys listed in `required_keys` are present in the
context before `run()` is called.  If any key is absent, it raises `ValueError`
with a descriptive message listing the missing keys.

This is the earliest possible point to catch misconfigured pipelines — before any
expensive computation happens.

In [ ]:
class TextProcessorAgent(InheritableAgent):
    """Requires 'input' and 'language' keys."""
    def run(self, ctx):
        return {
            'processed': ctx['input'].upper(),
            'language': ctx['language'],
        }


ValidatedTextAgent = build_agent(
    TextProcessorAgent,
    ValidatingTrait,
    validate_kwargs={'required_keys': ['input', 'language']},
)

validated_agent = ValidatedTextAgent()

# Case 1: all required keys present — should succeed
ok_result = validated_agent.execute({'input': 'hello world', 'language': 'en'})
print(f'Success case: {ok_result}')
assert ok_result['processed'] == 'HELLO WORLD'

print()

# Case 2: missing 'language' — should raise ValueError
try:
    validated_agent.execute({'input': 'hello world'})
    print('ERROR: should have raised ValueError')
except ValueError as exc:
    print(f'Missing one key → ValueError: {exc}')
    assert 'language' in str(exc).lower() or 'missing' in str(exc).lower()

# Case 3: both keys missing — should raise ValueError
try:
    validated_agent.execute({})
    print('ERROR: should have raised ValueError')
except ValueError as exc:
    print(f'Missing both keys → ValueError: {exc}')

print()
print('ValidatingTrait verified: ValueError on missing required keys.')

## Section 9 — build_agent(): Combining Multiple Traits

`build_agent(BaseClass, Trait1, Trait2, ...)` calls `type()` internally to construct
a new class whose MRO is `[ComposedClass, Trait1, Trait2, ..., BaseClass, ...]`.

The MRO order matters: traits listed first wrap traits listed later.  So with
`build_agent(MyAgent, LoggingTrait, TimingTrait, CachingTrait)` the execution flow is:

```
LoggingTrait.pre_run
  → TimingTrait wraps run()
    → CachingTrait may short-circuit run()
      → MyAgent.run()
```

We verify the MRO and confirm all three traits are active simultaneously.

In [ ]:
multi_run_count = {'count': 0}

class CoreAgent(InheritableAgent):
    def run(self, ctx):
        multi_run_count['count'] += 1
        return {'value': ctx.get('x', 0) * 2, 'call_number': multi_run_count['count']}


# Combine three traits at once
SuperAgent = build_agent(
    CoreAgent,
    LoggingTrait,
    TimingTrait,
    CachingTrait,
    cache_kwargs={'cache_keys': ['x']},
)

print(f'Composed class: {SuperAgent.__name__}')
print('MRO:')
for i, cls in enumerate(SuperAgent.__mro__):
    print(f'  [{i}] {cls.__name__}')
print()

# Verify all three traits are in the MRO
mro_names = [c.__name__ for c in SuperAgent.__mro__]
assert 'LoggingTrait' in mro_names
assert 'TimingTrait'  in mro_names
assert 'CachingTrait' in mro_names
print('All three traits present in MRO.')
print()

super_agent = SuperAgent()

# First call — cache miss, real run, timing measured
r1 = super_agent.execute({'x': 5})
print(f'Call 1 (x=5): {r1}')
assert '_duration_ms' in r1, 'TimingTrait must inject _duration_ms'

# Second call same x — cache hit, run() NOT called again
r2 = super_agent.execute({'x': 5})
print(f'Call 2 (x=5): {r2}  (cache hit — run() not called)')

# Third call different x — cache miss
r3 = super_agent.execute({'x': 10})
print(f'Call 3 (x=10): {r3}')

print()
assert multi_run_count['count'] == 2, f'Expected 2 real run() calls, got {multi_run_count["count"]}'
print('build_agent() multi-trait composition verified.')

## Section 10 — @mixin Decorator

The `@mixin(*traits)` decorator applies traits to an existing class **without
requiring you to change the class definition**.  It is equivalent to calling
`build_agent` but works as a decorator on the class definition site.

This is useful when:
- You receive a third-party agent class and cannot edit it.
- You want to conditionally apply traits in test vs. production environments.
- You prefer the decorator style over the factory function style.

In [ ]:
mixin_run_count = {'count': 0}

@mixin(TimingTrait, LoggingTrait)
class MixinAgent(InheritableAgent):
    """Agent with traits applied via @mixin decorator."""
    def run(self, ctx):
        mixin_run_count['count'] += 1
        return {'mixed': True, 'value': ctx.get('n', 0) + 100}


print(f'Class name after @mixin: {MixinAgent.__name__}')
print('MRO:')
for i, cls in enumerate(MixinAgent.__mro__):
    print(f'  [{i}] {cls.__name__}')
print()

mixin_names = [c.__name__ for c in MixinAgent.__mro__]
assert 'TimingTrait'  in mixin_names, 'TimingTrait missing from MRO'
assert 'LoggingTrait' in mixin_names, 'LoggingTrait missing from MRO'

ma = MixinAgent()
mixin_result = ma.execute({'n': 7})
print(f'Result: {mixin_result}')
assert '_duration_ms' in mixin_result
assert mixin_result['value'] == 107
print()
print('@mixin decorator verified: traits active without modifying class definition.')

## Section 11 — @overload: Single-Dispatch by Argument Type

The `@overload` decorator (distinct from `typing.overload`) enables Python-style
**single dispatch**: different implementations of a function are selected based on
the runtime type of the first argument.

This is useful for agent `run()` methods that need to handle multiple input
data types naturally — e.g., an agent that can process a number, a string, or a
list without a chain of `isinstance` checks.

In [ ]:
from multigen.inheritance import overload

class TypeRouter:
    @overload
    def process(self, value):
        """Default/fallback — called when no type match is found."""
        return {'type': 'unknown', 'value': repr(value)}

    @process.register(int)
    def process_int(self, value):
        return {'type': 'int', 'doubled': value * 2}

    @process.register(str)
    def process_str(self, value):
        return {'type': 'str', 'upper': value.upper(), 'length': len(value)}

    @process.register(list)
    def process_list(self, value):
        return {'type': 'list', 'sorted': sorted(value), 'count': len(value)}


router = TypeRouter()

cases = [
    42,
    'hello multigen',
    [3, 1, 4, 1, 5, 9],
    3.14,        # float — no overload registered, hits default
]

print('Overload dispatch results:')
for val in cases:
    result = router.process(val)
    print(f'  process({val!r:30}) → {result}')

print()
assert router.process(10)['doubled']  == 20
assert router.process('hi')['upper']  == 'HI'
assert router.process([2, 1])['sorted'] == [1, 2]
assert router.process(3.14)['type'] == 'unknown'   # fallback
print('@overload dispatch verified.')

## Section 12 — MultiMethod: N-ary Dispatch with Fallback

`MultiMethod` extends dispatch beyond a single argument. You register handlers
keyed on **tuples of types**, and `MultiMethod.dispatch(*args)` selects the most
specific registered handler.

A fallback handler registered as `MultiMethod.register_fallback(fn)` catches any
combination that has no explicit registration.

In [ ]:
mm = MultiMethod(name='combine')

# Register handlers for specific type pairs
mm.register((int, int),   lambda a, b: {'op': 'int+int',   'result': a + b})
mm.register((str, str),   lambda a, b: {'op': 'str+str',   'result': a + ' ' + b})
mm.register((list, list), lambda a, b: {'op': 'list+list', 'result': a + b})
mm.register((int, str),   lambda a, b: {'op': 'int+str',   'result': f'{a}: {b}'})

# Fallback for unregistered combinations
mm.register_fallback(lambda *args: {'op': 'fallback', 'types': [type(a).__name__ for a in args]})

test_cases = [
    (3, 4),
    ('hello', 'world'),
    ([1, 2], [3, 4]),
    (7, 'items'),
    (3.14, True),   # no registration → fallback
]

print('MultiMethod dispatch results:')
for args in test_cases:
    result = mm.dispatch(*args)
    type_sig = tuple(type(a).__name__ for a in args)
    print(f'  dispatch{args!r:30} → {result}')

print()
assert mm.dispatch(2, 3)['result']       == 5
assert mm.dispatch('a', 'b')['result']   == 'a b'
assert mm.dispatch([1], [2])['result']   == [1, 2]
assert mm.dispatch(5, 'x')['result']     == '5: x'
assert mm.dispatch(1.0, True)['op']      == 'fallback'
print('MultiMethod n-ary dispatch verified.')

## Section 13 — ShapeRegistry + PolymorphicAgent

A `PolymorphicAgent` can change its execution strategy at runtime based on context.
You register named **shapes** (concrete agent implementations) in a `ShapeRegistry`,
then provide a `morph_fn` — a callable that inspects `ctx` and returns the name of
the shape to use.

This is the agent equivalent of the Strategy pattern: the same logical agent
transparently switches between a fast approximate strategy and a thorough exact
strategy depending on the caller's requirements.

In [ ]:
# Define two concrete strategy agents

class FastSearchAgent(InheritableAgent):
    """Quick approximate search — low latency."""
    def run(self, ctx):
        return {
            'strategy': 'fast',
            'results': [f'approx_result_{i}' for i in range(3)],
            'latency': 'low',
        }


class ThoroughSearchAgent(InheritableAgent):
    """Exhaustive exact search — higher latency."""
    def run(self, ctx):
        return {
            'strategy': 'thorough',
            'results': [f'exact_result_{i}' for i in range(10)],
            'latency': 'high',
        }


# Build the registry
registry = ShapeRegistry()
registry.register('fast',      FastSearchAgent())
registry.register('thorough',  ThoroughSearchAgent())

# morph_fn: inspect ctx['mode'] to choose shape
def select_shape(ctx):
    return 'thorough' if ctx.get('mode') == 'thorough' else 'fast'


poly = PolymorphicAgent(registry=registry, morph_fn=select_shape)

print('Registered shapes:', registry.shapes())
print()

# Run in fast mode
fast_result = poly.execute({'mode': 'fast', 'query': 'multigen'})
print(f'fast  mode: {fast_result}')
assert fast_result['strategy'] == 'fast'
assert len(fast_result['results']) == 3

# Run in thorough mode
thorough_result = poly.execute({'mode': 'thorough', 'query': 'multigen'})
print(f'thorough mode: {thorough_result}')
assert thorough_result['strategy'] == 'thorough'
assert len(thorough_result['results']) == 10

print()
print('ShapeRegistry + PolymorphicAgent verified: morph_fn selects strategy at runtime.')

## Section 14 — TypeAdapter: Bidirectional Type Conversion Registry

`TypeAdapter` maintains a registry of conversion functions between pairs of types.
It exposes:

- `register(from_type, to_type, fn)` — register a converter.
- `can_adapt(value, to_type)` — check whether a conversion exists.
- `adapt(value, to_type)` — execute the conversion.

This is useful for agent pipelines where upstream agents emit one type and
downstream agents expect another.

In [ ]:
adapter = TypeAdapter()

# Register int → str and str → int converters
adapter.register(int, str, lambda x: str(x))
adapter.register(str, int, lambda x: int(x))
adapter.register(list, str, lambda x: ','.join(str(i) for i in x))

# Test can_adapt
print('can_adapt checks:')
print(f'  can_adapt(42, str)   : {adapter.can_adapt(42, str)}')
print(f'  can_adapt("7", int)  : {adapter.can_adapt("7", int)}')
print(f'  can_adapt([1,2], str): {adapter.can_adapt([1, 2], str)}')
print(f'  can_adapt(3.14, int) : {adapter.can_adapt(3.14, int)}')
print()

assert adapter.can_adapt(42, str)        is True
assert adapter.can_adapt('7', int)       is True
assert adapter.can_adapt([1, 2], str)    is True
assert adapter.can_adapt(3.14, int)      is False   # no float→int registered

# Test adapt
print('adapt results:')
r_int_str = adapter.adapt(99, str)
r_str_int = adapter.adapt('42', int)
r_list_str = adapter.adapt([1, 2, 3], str)

print(f'  adapt(99, str)       → {r_int_str!r}  (type: {type(r_int_str).__name__})')
print(f'  adapt("42", int)     → {r_str_int!r}  (type: {type(r_str_int).__name__})')
print(f'  adapt([1,2,3], str)  → {r_list_str!r} (type: {type(r_list_str).__name__})')
print()

assert r_int_str == '99'
assert r_str_int == 42
assert r_list_str == '1,2,3'

# adapt on unregistered pair should raise
try:
    adapter.adapt(3.14, int)
    print('ERROR: should have raised')
except (TypeError, KeyError, ValueError) as exc:
    print(f'adapt on unregistered pair → {type(exc).__name__}: {exc}')

print()
print('TypeAdapter verified: can_adapt() and adapt() work correctly.')

## Section 15 — DynamicAgent.build(): Runtime Class Construction

`DynamicAgent.build(base, mixins=[...])` constructs a new agent class at runtime
from a base class and an arbitrary list of mixin/trait classes.  The class is
assembled using `type()` and returned as a first-class Python class object —
it can be instantiated, inspected, and passed around like any other class.

This is the lowest-level API for programmatic class construction, useful when
the set of traits is not known until runtime (e.g., driven by a configuration file
or user preference).

In [ ]:
class RawAgent(InheritableAgent):
    def run(self, ctx):
        return {'raw': True, 'input': ctx.get('data')}


# Build at runtime with a dynamically determined mixin list
trait_config = [TimingTrait, LoggingTrait]   # could come from a config file

DynClass = DynamicAgent.build(base=RawAgent, mixins=trait_config)

print(f'Dynamically built class  : {DynClass}')
print(f'Class name               : {DynClass.__name__}')
print('MRO:')
for i, cls in enumerate(DynClass.__mro__):
    print(f'  [{i}] {cls.__name__}')
print()

# Instantiate and run
dyn_instance = DynClass()
dyn_result = dyn_instance.execute({'data': 'hello dynamic world'})
print(f'Result: {dyn_result}')
print()

assert '_duration_ms' in dyn_result, 'TimingTrait must inject _duration_ms'
assert dyn_result['raw'] is True
assert dyn_result['input'] == 'hello dynamic world'

# Build a second class with different traits
DynClassV2 = DynamicAgent.build(base=RawAgent, mixins=[CachingTrait],
                                 cache_kwargs={'cache_keys': ['data']})
print(f'Second dynamic class MRO: {[c.__name__ for c in DynClassV2.__mro__]}')
assert 'CachingTrait' in [c.__name__ for c in DynClassV2.__mro__]

print()
print('DynamicAgent.build() verified: classes constructed from runtime mixin lists.')

## Summary

### Inheritance and trait API at a glance

| API | Description |
|-----|-------------|
| `InheritableAgent` | Base class with `pre_run` / `run` / `post_run` / `on_error` hooks |
| `Trait` | Base mixin; all built-in traits extend it |
| `LoggingTrait` | Emits DEBUG log lines before and after `run()` |
| `RetryTrait(max_retries=N)` | Retries `run()` up to N times on exception |
| `CachingTrait(cache_keys=[...])` | Memoise `run()` results keyed on context fields |
| `TimingTrait` | Injects `_duration_ms` into the result dict |
| `ValidatingTrait(required_keys=[...])` | Raises `ValueError` if keys missing from context |
| `build_agent(Base, T1, T2, ...)` | Compose a new class via `type()`, respecting MRO |
| `@mixin(T1, T2, ...)` | Decorator-style trait application |
| `@overload` | Single-dispatch by first-argument type |
| `MultiMethod` | N-ary dispatch on tuple-of-types with fallback |
| `PolymorphicAgent(registry, morph_fn)` | Runtime strategy selection via `ShapeRegistry` |
| `TypeAdapter` | Bidirectional type conversion registry |
| `DynamicAgent.build(base, mixins)` | Programmatic class construction at runtime |

### Next steps

- **Notebook 28**: Performance optimization and advanced memory systems.
- Combine `RetryTrait + CachingTrait` to avoid retrying already-cached results.
- Use `TypeAdapter` in pipeline connectors to enforce type contracts between agents.
- Drive `DynamicAgent.build()` from a YAML/JSON configuration for declarative
  agent composition.